<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Tesis_v10_Sandbox_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background-color: #1a3e59; padding: 20px; border-radius: 10px;">
<h1 style="color: #ffffff; margin: 0; text-align: center;">Tesis: Análisis MCA de Deudores BCRA vs Padrón ARCA 🚀</h1>
<p style="color: #d1e8ff; text-align: center; margin-top: 10px; font-size: 1.1em;">Fase 2: Reducción de Dimensionalidad, Métricas de Data Loss, Tablas de Contingencia y MCA Micro/Macro.</p>
</div>


## <span style="color: #2b7a78;">1. Instalación de Librerías y Configuración Inicial ⚙️</span>


In [1]:
!pip install prince plotly polars pandas numpy kaleido
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import prince
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.1 MB/s eta 0:00:00
Mounted at /content/drive


## <span style="color: #2b7a78;">2. Carga de Bases de Datos y Limpieza Poblacional 🧹</span>


In [ ]:
# ==========================================
# ⚙️ PASO 1: FIJANDO CEROS A LA IZQUIERDA Y REPORTE DE PÉRDIDAS EN CASCADA
# ==========================================
print("1. Cargando bases de datos en modo Lazy...")
df_bcra = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/deudores_enero_2026_clasificados.parquet")
padron = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/padron_fisicas_enero.parquet")
entidades_maestro = pl.scan_csv("/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv")

print("Estandarizando estructuras de IDs y Códigos...")
df_bcra = df_bcra.with_columns([
    pl.col("nro_id").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
])
padron = padron.with_columns([
    pl.col("cuit").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("fecha_fallecimiento").str.strip_chars().alias("fallecimiento_clean"),
    pl.col("fecha_nacimiento").str.strip_chars().alias("nacimiento_clean"),
    pl.col("sexo").str.strip_chars().alias("sexo_clean"),
    pl.col("provincia").cast(pl.Utf8).str.zfill(2)
])
entidades_maestro = entidades_maestro.with_columns(
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
)

print("Ejecutando cruce poblacional (BCRA + Padrón)...")
joined = df_bcra.join(padron, left_on="nro_id", right_on="cuit", how="left")

# ==========================================
# 📊 CÁLCULO DE MÉTRICAS DEL EMBUDO EN CASCADA (DATA LOSS)
# ==========================================
metricas = joined.select([
    pl.len().alias("total_bcra"),
    pl.col("sexo_clean").is_null().sum().alias("perdidos_join"),
    (pl.col("sexo_clean").is_not_null() & ~pl.col("sexo_clean").is_in(['M', 'F'])).sum().alias("perdidos_genero"),
    (pl.col("sexo_clean").is_in(['M', 'F']) &
     ~(((pl.col("fallecimiento_clean") == "") | pl.col("fallecimiento_clean").is_null()) &
       (pl.col("nacimiento_clean") != "1901-01-01"))
    ).sum().alias("perdidos_vital"),
    (pl.col("sexo_clean").is_in(['M', 'F']) &
     ((pl.col("fallecimiento_clean") == "") | pl.col("fallecimiento_clean").is_null()) &
     (pl.col("nacimiento_clean") != "1901-01-01") &
     (pl.col("provincia").is_null() | (pl.col("provincia").str.strip_chars() == ""))
    ).sum().alias("perdidos_provincia")
]).collect().to_pandas()

total_bcra = metricas['total_bcra'][0]
perdidos_join = metricas['perdidos_join'][0]
perdidos_genero = metricas['perdidos_genero'][0]
perdidos_vital = metricas['perdidos_vital'][0]
perdidos_provincia = metricas['perdidos_provincia'][0]
total_cruzados = total_bcra - perdidos_join
total_final = total_cruzados - perdidos_genero - perdidos_vital - perdidos_provincia

print("\n" + "="*65)
print("📊 REPORTE DE PÉRDIDA Y LIMPIEZA DE FILAS (EMBUDO)")
print("="*65)
print(f"1. Total de registros originales BCRA  : {total_bcra:,}")
print(f"   [-] Perdidos en JOIN (Sin Match)    : {perdidos_join:,} ({(perdidos_join/total_bcra)*100:.2f}%)")
print(f"2. Registros iniciales cruzados        : {total_cruzados:,}")
print(f"   [-] Excluidos: Género inv. (ej. X)  : {perdidos_genero:,} ({(perdidos_genero/total_cruzados)*100:.3f}%)")
print(f"   [-] Excluidos: Fallecidos/Fecha1901 : {perdidos_vital:,} ({(perdidos_vital/total_cruzados)*100:.3f}%)")
print(f"   [-] Excluidos: Sin Provincia        : {perdidos_provincia:,} ({(perdidos_provincia/total_cruzados)*100:.3f}%)")
print("-" * 65)
print(f"🏆 POBLACIÓN ACTIVA FINAL CONSERVADA   : {total_final:,}")
print("="*65 + "\n")

# ==========================================
# 🧹 APLICACIÓN FÍSICA DE LOS FILTROS
# ==========================================
base_limpia = joined.filter(
    pl.col("sexo_clean").is_in(['M', 'F'])
).filter(
    (pl.col("fallecimiento_clean") == "") | pl.col("fallecimiento_clean").is_null()
).filter(
    pl.col("nacimiento_clean") != "1901-01-01"
).filter(
    pl.col("provincia").is_not_null() & (pl.col("provincia").str.strip_chars() != "")
)

joined_con_nombres = base_limpia.join(entidades_maestro, on="cod_entidad", how="left")


1. Cargando bases de datos en modo Lazy...
Estandarizando estructuras de IDs y Códigos...
Ejecutando cruce poblacional (BCRA + Padrón)...


## <span style="color: #2b7a78;">3. Filtrado Top 20 y Macro-Regiones Geográficas (CABA Corregida) 🗺️</span>


In [ ]:
top_20_calculado = joined_con_nombres.group_by(["cod_entidad", "nombre_entidad", "grupo_entidad"]).agg(
    pl.col("deuda_total").sum().alias("volumen_deuda_total")
).sort("volumen_deuda_total", descending=True).limit(20)

df_top20_ranking = top_20_calculado.collect().to_pandas()
lista_codigos_top20 = df_top20_ranking["cod_entidad"].tolist()
base_filtrada_top20 = joined_con_nombres.filter(pl.col("cod_entidad").is_in(lista_codigos_top20))

base_features = base_filtrada_top20.with_columns(
    pl.col("nacimiento_clean").str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("fecha_nac_dt"),
    pl.col("codigo_postal").cast(pl.Utf8).str.extract(r"(\d{4})").cast(pl.Int32, strict=False).fill_null(0).alias("cp_num")
).with_columns(
    ((pl.date(2026, 1, 1) - pl.col("fecha_nac_dt")).dt.total_days() / 365.25).floor().alias("edad")
).with_columns([
    pl.when(pl.col("edad") <= 25).then(pl.lit("Jóvenes (18-25)"))
    .when(pl.col("edad") <= 40).then(pl.lit("Adultos en Inserción (26-40)"))
    .when(pl.col("edad") <= 65).then(pl.lit("Adultos Consolidados (41-65)"))
    .otherwise(pl.lit("Adultos Mayores (>65)")).alias("Edad"),

    pl.when((pl.col("provincia") == "00") | ((pl.col("cp_num") >= 1000) & (pl.col("cp_num") <= 1499))).then(pl.lit("AMBA - CABA"))
    .when((pl.col("cp_num") >= 1500) & (pl.col("cp_num") <= 1999)).then(pl.lit("AMBA - Conurbano"))
    .when(pl.col("provincia").is_in(["01", "03", "05", "11", "19"])).then(pl.lit("Región Pampeana"))
    .when(pl.col("provincia").is_in(["02", "06", "08", "12", "13", "23"])).then(pl.lit("NOA"))
    .when(pl.col("provincia").is_in(["04", "14", "16", "17"])).then(pl.lit("NEA"))
    .when(pl.col("provincia").is_in(["07", "09", "10"])).then(pl.lit("Cuyo"))
    .when(pl.col("provincia").is_in(["15", "18", "20", "21", "22"])).then(pl.lit("Patagonia"))
    .otherwise(pl.lit("Sin Especificar")).alias("Geografia")
]).select([
    pl.col("nombre_entidad").alias("nombre_entidad"),
    pl.col("sexo_clean").alias("sexo_clean"),
    pl.col("Edad"),
    pl.col("Geografia"),
    pl.col("deuda_total")
]).drop_nulls()

df_eda = base_features.collect().to_pandas()
print("\nBase materializada y lista.")


## <span style="color: #2b7a78;">4. Análisis Estadístico: Cuantiles de Deuda cruzados con Demografía 💰</span>
Calculamos 4 cuantiles de deuda de la población para separar la cartera total en 4 volúmenes equitativos. Luego, presentamos la tabla cruzada de los cuantiles vs la geografía y edad.

In [ ]:
df_q = df_eda.copy()

# 1. Creación de Cuantiles de Deuda
df_q['Cuantil_Deuda'], bins_deuda = pd.qcut(df_q['deuda_total'], q=4, retbins=True,
                                            labels=['Q1 (Micro)', 'Q2 (Baja)', 'Q3 (Media)', 'Q4 (Alta)'])

print("="*60)
print("📊 REPORTE DE LÍMITES - CUANTILES DE DEUDA TOTAL")
print("="*60)
print(f"🔹 Q1 (Micro) : $0 hasta ${bins_deuda[1]:,.2f}")
print(f"🔹 Q2 (Baja)  : ${bins_deuda[1]:,.2f} hasta ${bins_deuda[2]:,.2f}")
print(f"🔹 Q3 (Media) : ${bins_deuda[2]:,.2f} hasta ${bins_deuda[3]:,.2f}")
print(f"🔹 Q4 (Alta)  : Más de ${bins_deuda[3]:,.2f}")
print("="*60 + "\n")

# 2. Estadísticas Descriptivas de los Cuantiles
print("🎯 PERFIL DEMOGRÁFICO DE LOS CUANTILES DE DEUDA (En %)\n")

print("--- Distribución de GEOGRAFÍA por Cuantil de Deuda ---")
tab_geo = pd.crosstab(df_q['Cuantil_Deuda'], df_q['Geografia'], normalize='index') * 100
print(tab_geo.round(1).to_string())
print("\n--- Distribución de EDAD por Cuantil de Deuda ---")
tab_edad = pd.crosstab(df_q['Cuantil_Deuda'], df_q['Edad'], normalize='index') * 100
print(tab_edad.round(1).to_string())
print("-" * 60)


## <span style="color: #2b7a78;">5. Análisis Estadístico: Desglose por Entidad Específica (Top 20) 🏢</span>
Realizamos una exploración analítica descriptiva cruzando los 20 bancos y fintechs principales con las variables demográficas.

In [ ]:
print("="*60)
print("🎯 PERFIL DEMOGRÁFICO POR ENTIDAD FINANCIERA (TOP 20 - En %)")
print("="*60)

# Obtenemos la cantidad total de deudores por entidad para ordenarlas de mayor a menor
orden_entidades = df_eda['nombre_entidad'].value_counts().index

print("\n--- Distribución de GEOGRAFÍA por Banco ---")
tab_geo_banco = pd.crosstab(df_eda['nombre_entidad'], df_eda['Geografia'], normalize='index').reindex(orden_entidades) * 100
print(tab_geo_banco.round(1).to_string())

print("\n--- Distribución de EDAD por Banco ---")
tab_edad_banco = pd.crosstab(df_eda['nombre_entidad'], df_eda['Edad'], normalize='index').reindex(orden_entidades) * 100
print(tab_edad_banco.round(1).to_string())
print("-" * 60)


## <span style="color: #2b7a78;">6. Reducción de Dimensionalidad (MCA MICRO: Top 20 Bancos sin agrupar) 🧠</span>
Entrenamos el modelo SVD dejando a cada banco como una clase independiente para observar sus coordenadas y distancias geométricas en el plano sin intervención analítica macro.

In [ ]:
df_mca_micro = df_eda[
    (df_eda['sexo_clean'].isin(['M', 'F'])) &
    (df_eda['Geografia'] != 'Sin Especificar')
].copy()

X_micro = df_mca_micro[['nombre_entidad', 'sexo_clean', 'Edad', 'Geografia']].copy()
X_micro.columns = ['Entidad_Especifica', 'Genero', 'Edad', 'Geografia']

# Muestreo Estratificado por entidad (manteniendo la cuota de mercado exacta)
fraccion_micro = min(250000 / len(X_micro), 1.0)
df_sample_micro = X_micro.groupby('Entidad_Especifica', group_keys=False).apply(lambda x: x.sample(frac=fraccion_micro, random_state=42))

print("\nEntrenando modelo MCA Micro (Con Nube Top 20)...")
mca_micro = prince.MCA(n_components=2, n_iter=10, random_state=42)
mca_micro = mca_micro.fit(df_sample_micro)

v1_micro = mca_micro.percentage_of_variance_[0]
v2_micro = mca_micro.percentage_of_variance_[1]

coords_micro = mca_micro.column_coordinates(df_sample_micro).copy()
coords_micro.columns = ['Dim_1', 'Dim_2']
coords_micro['Atributo_Original'] = coords_micro.index

def clasificar_variable_micro(attr):
    if str(attr).startswith('Entidad_Especifica_'): return 'Banco / Entidad (Top 20)'
    if str(attr).startswith('Edad_'): return 'Rango Etario'
    if str(attr).startswith('Geografia_'): return 'Región Geográfica'
    if str(attr).startswith('Genero_'): return 'Género'
    return 'Otro'

coords_micro['Categoria'] = coords_micro['Atributo_Original'].apply(lambda x: str(x).split('_', 1)[-1] if '_' in str(x) else x)
coords_micro['Tipo'] = coords_micro['Atributo_Original'].apply(clasificar_variable_micro)

fig_micro = px.scatter(
    coords_micro, x='Dim_1', y='Dim_2', color='Tipo', text='Categoria',
    title='Mapa Perceptual MCA - Análisis Micro (Nube de Top 20 Entidades)',
    labels={'Dim_1': f"Dimensión 1 ({v1_micro:.2f}%)", 'Dim_2': f"Dimensión 2 ({v2_micro:.2f}%)"},
    color_discrete_sequence=px.colors.qualitative.Dark24
)
fig_micro.update_traces(textposition='top center', marker=dict(size=14, opacity=0.85, line=dict(width=1, color='Black')))
fig_micro.update_layout(template='plotly_white', width=1400, height=900)
fig_micro.show()


## <span style="color: #2b7a78;">7. Exportación a HTML y Hosting Local 🚀</span>


In [ ]:
archivo_html_micro = "Mapa_MCA_Micro_Top20.html"
fig_micro.write_html(archivo_html_micro)
print(f"¡Gráfico interactivo guardado: {archivo_html_micro}!")

import threading
import http.server
import socketserver
import os
from google.colab import output

PORT = 8083
os.chdir('/content')

def iniciar_servidor():
    Handler = http.server.SimpleHTTPRequestHandler
    socketserver.TCPServer.allow_reuse_address = True
    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        print(f"\n[INFO] Servidor web local en puerto {PORT}.")
        httpd.serve_forever()

threading.Thread(target=iniciar_servidor, daemon=True).start()
print("Generando enlaces seguros de visualización...")
output.serve_kernel_port_as_window(PORT, path='/Mapa_MCA_Micro_Top20.html')
